# Zero Dollar Lost Analysis

This notebook analyzes the parametric study results to find parameter combinations that result in **zero dollar lost**.

## Objectives:
- Load and process the large CSV files
- Filter for combinations where Dollar_Lost = 0
- Analyze which parameter values lead to zero losses
- Visualize the optimal parameter space

## Data Sources:
- `outputs/parametric_study_results_taup=0.1.csv`
- `outputs/parametric_study_results_taup=1.csv`

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

print(f"📚 Libraries loaded successfully")
print(f"🐍 Python {pd.__version__} | NumPy {np.__version__} | Pandas {pd.__version__}")

📚 Libraries loaded successfully
🐍 Python 2.3.2 | NumPy 2.3.3 | Pandas 2.3.2


In [2]:
# Data processing functions - handle units and arrays properly

def extract_numeric_from_units(value):
    """Extract numeric value from strings with units, preserving infinite values"""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)  # Keep inf as inf, not 0
    
    str_val = str(value)
    if 'inf' in str_val.lower():
        return np.inf if '-inf' not in str_val.lower() else -np.inf
    
    match = re.search(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', str_val)
    if match:
        try:
            result = float(match.group())
            return result
        except (ValueError, OverflowError):
            return np.nan
    return np.nan  # Return NaN, not 0!

def extract_scalar_from_array(value):
    """Extract final scalar value from array-like strings"""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)  # Keep inf as inf, not 0
    
    str_val = str(value).strip()
    if 'inf' in str_val.lower():
        return np.inf if '-inf' not in str_val.lower() else -np.inf
    
    # Handle array-like strings
    if str_val.startswith('['):
        numbers = re.findall(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', str_val)
        if numbers:
            try:
                result = float(numbers[-1])  # Take last value (final state)
                return result
            except (ValueError, OverflowError):
                return np.nan
        return np.nan
    
    # Otherwise extract the first number
    match = re.search(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', str_val)
    if match:
        try:
            result = float(match.group())
            return result
        except (ValueError, OverflowError):
            return np.nan
    return np.nan

print("✅ Data processing functions defined")

✅ Data processing functions defined


In [3]:
# Load and process the CSV files

DATA_DIR = Path('outputs')
CSV_A = DATA_DIR / 'parametric_study_results_taup=0.1.csv'
CSV_B = DATA_DIR / 'parametric_study_results_taup=1.csv'

print("📂 Loading CSV files...")
print(f"   File A: {CSV_A.exists()} - {CSV_A}")
print(f"   File B: {CSV_B.exists()} - {CSV_B}")

if not CSV_A.exists() or not CSV_B.exists():
    raise FileNotFoundError("One or both CSV files not found!")

# Load the data
df_a = pd.read_csv(CSV_A)
df_b = pd.read_csv(CSV_B)

print(f"📊 Data loaded:")
print(f"   τₚ = 0.1s: {df_a.shape[0]:,} rows × {df_a.shape[1]} columns")
print(f"   τₚ = 1.0s: {df_b.shape[0]:,} rows × {df_b.shape[1]} columns")

# Add source identifier
df_a['tau_p_source'] = 0.1
df_b['tau_p_source'] = 1.0

# Combine datasets
df_combined = pd.concat([df_a, df_b], ignore_index=True)
print(f"🔄 Combined dataset: {df_combined.shape[0]:,} rows × {df_combined.shape[1]} columns")

# Show sample of columns
print(f"\n📋 Available columns: {list(df_combined.columns[:10])}...")

📂 Loading CSV files...
   File A: True - outputs/parametric_study_results_taup=0.1.csv
   File B: True - outputs/parametric_study_results_taup=1.csv
📊 Data loaded:
   τₚ = 0.1s: 71,850 rows × 37 columns
   τₚ = 1.0s: 72,600 rows × 37 columns
🔄 Combined dataset: 144,450 rows × 38 columns

📋 Available columns: ['V_plasma (m^3)', 'tau_p_T (s)', 'tau_p_He3 (s)', 'P_aux (MW)', 'P_aux_all_DT (MW)', 'P_lost_rad (MW)', 'P_lost_rad_all_DT (MW)', 'T_i (keV)', 'n_tot (m^-3)', 'TBR_DT']...


In [4]:
# Process the Dollar_Lost column to find zero values

print("💰 Processing Dollar_Lost column...")

# Check if Dollar_Lost column exists
dollar_col = 'Dollar_Lost ($)'
if dollar_col not in df_combined.columns:
    print(f"❌ Column '{dollar_col}' not found!")
    print(f"Available columns containing 'dollar' or 'cost': {[col for col in df_combined.columns if 'dollar' in col.lower() or 'cost' in col.lower()]}")
    raise ValueError(f"Dollar_Lost column not found")

print(f"✅ Found Dollar_Lost column: {dollar_col}")

# Sample the raw data to understand format
print(f"\n🔍 Sample raw Dollar_Lost values:")
for i in range(min(5, len(df_combined))):
    sample_val = df_combined[dollar_col].iloc[i]
    print(f"   [{i}]: {str(sample_val)[:100]}...")

# Process Dollar_Lost values
print(f"\n⚙️  Converting Dollar_Lost to numeric...")
df_combined['Dollar_Lost_processed'] = df_combined[dollar_col].apply(extract_scalar_from_array)

# Check for successful conversion
n_total = len(df_combined)
n_finite = np.isfinite(df_combined['Dollar_Lost_processed']).sum()
n_inf = np.isinf(df_combined['Dollar_Lost_processed']).sum()
n_nan = np.isnan(df_combined['Dollar_Lost_processed']).sum()

print(f"📈 Conversion results:")
print(f"   Total values: {n_total:,}")
print(f"   Finite values: {n_finite:,} ({100*n_finite/n_total:.1f}%)")
print(f"   Infinite values: {n_inf:,} ({100*n_inf/n_total:.1f}%)")
print(f"   NaN values: {n_nan:,} ({100*n_nan/n_total:.1f}%)")

if n_finite == 0:
    raise ValueError("No finite Dollar_Lost values found after processing!")

💰 Processing Dollar_Lost column...
✅ Found Dollar_Lost column: Dollar_Lost ($)

🔍 Sample raw Dollar_Lost values:
   [0]: [-8974821.917004162 -7760673.935040082 -6546525.953076003 -5332377.971111924 -4118229.9891478447 -29...
   [1]: [-15863301.313672652 -13717253.68672286 -11571206.059773065 -9425158.432823274 -7279110.80587348 -51...
   [2]: [-21391693.007580172 -18497743.56366537 -15603794.119750567 -12709844.675835768 -9815895.231920965 -...
   [3]: [-26920084.70148769 -23278233.440607876 -19636382.17972807 -15994530.918848261 -12352679.65796845 -8...
   [4]: [-33808564.09815618 -29234813.192290653 -24661062.28642513 -20087311.38055961 -15513560.474694083 -1...

⚙️  Converting Dollar_Lost to numeric...
📈 Conversion results:
   Total values: 144,450
   Finite values: 144,450 (100.0%)
   Infinite values: 0 (0.0%)
   NaN values: 0 (0.0%)


In [5]:
# Find parameter combinations that give zero dollar lost

print("🎯 Searching for ZERO dollar lost combinations...")

# Filter for finite values first
df_finite = df_combined[np.isfinite(df_combined['Dollar_Lost_processed'])].copy()
print(f"📊 Working with {len(df_finite):,} finite Dollar_Lost values")

# Find zero dollar lost (with small tolerance for floating point precision)
tolerance = 1e-6  # Very small tolerance
zero_mask = np.abs(df_finite['Dollar_Lost_processed']) <= tolerance
df_zero_dollar = df_finite[zero_mask].copy()

print(f"\n💎 ZERO DOLLAR LOST RESULTS:")
print(f"   Found {len(df_zero_dollar):,} combinations with zero dollar lost!")
print(f"   That's {100*len(df_zero_dollar)/len(df_finite):.3f}% of finite combinations")

if len(df_zero_dollar) == 0:
    print("\n🔍 No exact zeros found. Let's check the smallest values:")
    smallest_losses = df_finite.nsmallest(10, 'Dollar_Lost_processed')
    print(smallest_losses[['Dollar_Lost_processed', 'tau_p_source']].to_string())
    
    # Try with larger tolerance
    tolerance = 1e3  # 1000 dollars
    near_zero_mask = np.abs(df_finite['Dollar_Lost_processed']) <= tolerance
    df_near_zero = df_finite[near_zero_mask].copy()
    print(f"\n📍 Near-zero results (≤ {tolerance:,.0f}$): {len(df_near_zero):,} combinations")
else:
    print(f"\n📋 Distribution by tau_p:")
    tau_p_dist = df_zero_dollar['tau_p_source'].value_counts().sort_index()
    for tau_p, count in tau_p_dist.items():
        print(f"   τₚ = {tau_p}s: {count:,} combinations")

🎯 Searching for ZERO dollar lost combinations...
📊 Working with 144,450 finite Dollar_Lost values

💎 ZERO DOLLAR LOST RESULTS:
   Found 9,150 combinations with zero dollar lost!
   That's 6.334% of finite combinations

📋 Distribution by tau_p:
   τₚ = 0.1s: 6,900 combinations
   τₚ = 1.0s: 2,250 combinations


In [6]:
# Analyze parameter combinations for zero dollar lost cases

if len(df_zero_dollar) > 0:
    print("🔬 ANALYZING ZERO DOLLAR LOST PARAMETER COMBINATIONS")
    print("="*60)
    
    # Key parameter columns to analyze
    key_params = [
        'V_plasma (m^3)', 'tau_p_T (s)', 'tau_p_He3 (s)', 'P_aux (MW)', 
        'T_i (keV)', 'n_tot (m^-3)', 'TBR_DT', 'TBR_DDn', 
        'eta_th', 'plant_avail', 'tau_p_source'
    ]
    
    # Filter to existing columns
    available_params = [col for col in key_params if col in df_zero_dollar.columns]
    print(f"📊 Analyzing {len(available_params)} key parameters: {available_params}")
    
    # Process parameter columns
    df_zero_processed = df_zero_dollar.copy()
    
    for col in available_params:
        if col == 'tau_p_source':
            continue  # Already processed
        
        print(f"\n⚙️  Processing {col}...")
        
        # Apply appropriate extraction function
        if col in ['Dollar_Lost ($)']:  # Array-like columns
            df_zero_processed[f'{col}_processed'] = df_zero_dollar[col].apply(extract_scalar_from_array)
        else:  # Unit-based columns
            df_zero_processed[f'{col}_processed'] = df_zero_dollar[col].apply(extract_numeric_from_units)
        
        # Check conversion success
        finite_count = np.isfinite(df_zero_processed[f'{col}_processed']).sum()
        print(f"   ✅ {finite_count}/{len(df_zero_processed)} values converted successfully")
    
    print("\n🎯 Zero dollar lost parameter analysis complete!")
    
else:
    print("⚠️  No zero dollar lost combinations found - skipping parameter analysis")
    print("   Consider analyzing near-zero combinations instead")

🔬 ANALYZING ZERO DOLLAR LOST PARAMETER COMBINATIONS
📊 Analyzing 11 key parameters: ['V_plasma (m^3)', 'tau_p_T (s)', 'tau_p_He3 (s)', 'P_aux (MW)', 'T_i (keV)', 'n_tot (m^-3)', 'TBR_DT', 'TBR_DDn', 'eta_th', 'plant_avail', 'tau_p_source']

⚙️  Processing V_plasma (m^3)...
   ✅ 9150/9150 values converted successfully

⚙️  Processing tau_p_T (s)...
   ✅ 9150/9150 values converted successfully

⚙️  Processing tau_p_He3 (s)...
   ✅ 9150/9150 values converted successfully

⚙️  Processing P_aux (MW)...
   ✅ 9150/9150 values converted successfully

⚙️  Processing T_i (keV)...
   ✅ 9150/9150 values converted successfully

⚙️  Processing n_tot (m^-3)...
   ✅ 9150/9150 values converted successfully

⚙️  Processing TBR_DT...
   ✅ 9150/9150 values converted successfully

⚙️  Processing TBR_DDn...
   ✅ 9150/9150 values converted successfully

⚙️  Processing eta_th...
   ✅ 9150/9150 values converted successfully

⚙️  Processing plant_avail...
   ✅ 9150/9150 values converted successfully

🎯 Zero doll

In [7]:
# Statistical summary of zero dollar lost parameter ranges

if len(df_zero_dollar) > 0:
    print("📊 STATISTICAL SUMMARY OF ZERO DOLLAR LOST PARAMETERS")
    print("="*60)
    
    # Create summary statistics
    processed_cols = [col for col in df_zero_processed.columns if col.endswith('_processed')]
    
    summary_stats = []
    
    for col in processed_cols:
        if col == 'Dollar_Lost ($)_processed':
            continue  # Skip the dollar lost column itself
        
        values = df_zero_processed[col]
        finite_values = values[np.isfinite(values)]
        
        if len(finite_values) > 0:
            original_col = col.replace('_processed', '')
            stats = {
                'Parameter': original_col,
                'Count': len(finite_values),
                'Min': finite_values.min(),
                'Max': finite_values.max(),
                'Mean': finite_values.mean(),
                'Std': finite_values.std(),
                'Unique_Values': len(finite_values.unique())
            }
            summary_stats.append(stats)
    
    # Convert to DataFrame and display
    summary_df = pd.DataFrame(summary_stats)
    
    if len(summary_df) > 0:
        print("\n📋 Parameter Ranges for Zero Dollar Lost Combinations:")
        print(summary_df.to_string(index=False, float_format='%.3f'))
        
        # Identify fixed vs variable parameters
        fixed_params = summary_df[summary_df['Unique_Values'] == 1]['Parameter'].tolist()
        variable_params = summary_df[summary_df['Unique_Values'] > 1]['Parameter'].tolist()
        
        print(f"\n🔒 Fixed parameters (same value for all zero-loss combinations): {len(fixed_params)}")
        for param in fixed_params:
            value = summary_df[summary_df['Parameter'] == param]['Min'].iloc[0]
            print(f"   • {param}: {value:.3f}")
        
        print(f"\n🔄 Variable parameters (different values allowed): {len(variable_params)}")
        for param in variable_params:
            row = summary_df[summary_df['Parameter'] == param].iloc[0]
            print(f"   • {param}: {row['Min']:.3f} to {row['Max']:.3f} ({row['Unique_Values']} unique values)")
    
    else:
        print("❌ No valid parameter statistics could be computed")

else:
    print("⚠️  Skipping statistical summary - no zero dollar lost combinations found")

📊 STATISTICAL SUMMARY OF ZERO DOLLAR LOST PARAMETERS

📋 Parameter Ranges for Zero Dollar Lost Combinations:
     Parameter  Count                       Min                       Max                      Mean                      Std  Unique_Values
   Dollar_Lost   9150                     0.000                     0.000                     0.000                    0.000              1
V_plasma (m^3)   9150                   150.000                   150.000                   150.000                    0.000              1
   tau_p_T (s)   9150                     0.100                     1.000                     0.321                    0.388              2
 tau_p_He3 (s)   9150                     1.000                     1.000                     1.000                    0.000              1
    P_aux (MW)   9150                    60.000                    60.000                    60.000                    0.000              1
     T_i (keV)   9150                    14.000     

In [8]:
# Visualize the zero dollar lost parameter space

if len(df_zero_dollar) > 0 and len(summary_df) > 0:
    print("📊 CREATING VISUALIZATIONS FOR ZERO DOLLAR LOST COMBINATIONS")
    print("="*60)
    
    # Create subplots for parameter distributions
    variable_params = summary_df[summary_df['Unique_Values'] > 1]['Parameter'].tolist()
    
    if len(variable_params) > 0:
        print(f"📈 Creating distribution plots for {len(variable_params)} variable parameters...")
        
        # Calculate subplot layout
        n_plots = len(variable_params)
        n_cols = min(3, n_plots)
        n_rows = (n_plots + n_cols - 1) // n_cols
        
        fig = make_subplots(
            rows=n_rows, cols=n_cols,
            subplot_titles=[param.replace(' (', '<br>(') for param in variable_params],
            vertical_spacing=0.1,
            horizontal_spacing=0.1
        )
        
        for i, param in enumerate(variable_params):
            row = (i // n_cols) + 1
            col = (i % n_cols) + 1
            
            # Get processed values
            processed_col = f'{param}_processed'
            if processed_col in df_zero_processed.columns:
                values = df_zero_processed[processed_col]
                finite_values = values[np.isfinite(values)]
                
                if len(finite_values) > 0:
                    # Create histogram
                    fig.add_trace(
                        go.Histogram(
                            x=finite_values,
                            name=param,
                            showlegend=False,
                            marker_color='lightblue',
                            opacity=0.7
                        ),
                        row=row, col=col
                    )
        
        fig.update_layout(
            title='Parameter Distributions for Zero Dollar Lost Combinations',
            height=300 * n_rows,
            showlegend=False
        )
        
        fig.show()
        
    else:
        print("ℹ️  All parameters are fixed - no distribution plots needed")
        
    # Create summary table visualization
    if len(fixed_params) > 0:
        print(f"\n📋 Fixed Parameter Values (Required for Zero Dollar Lost):")
        
        fixed_data = []
        for param in fixed_params:
            value = summary_df[summary_df['Parameter'] == param]['Min'].iloc[0]
            fixed_data.append([param, f"{value:.3f}"])
        
        fig_table = go.Figure(data=[go.Table(
            header=dict(values=['Parameter', 'Required Value'],
                       fill_color='lightblue',
                       align='left',
                       font=dict(size=12, color='black')),
            cells=dict(values=list(zip(*fixed_data)),
                      fill_color='white',
                      align='left',
                      font=dict(size=11))
        )])
        
        fig_table.update_layout(
            title='Fixed Parameters Required for Zero Dollar Lost',
            height=max(200, 50 + 30 * len(fixed_params))
        )
        
        fig_table.show()

else:
    print("⚠️  Skipping visualizations - no zero dollar lost combinations found")
    
    # Show distribution of dollar lost values instead
    print("\n📊 Showing distribution of Dollar Lost values instead...")
    
    finite_dollar_values = df_finite['Dollar_Lost_processed']
    
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=finite_dollar_values,
        nbinsx=50,
        name='Dollar Lost Distribution',
        marker_color='lightcoral',
        opacity=0.7
    ))
    
    fig.update_layout(
        title='Distribution of Dollar Lost Values',
        xaxis_title='Dollar Lost ($)',
        yaxis_title='Frequency',
        height=400
    )
    
    fig.show()
    
    # Show smallest values
    print(f"\n🏆 Top 10 smallest Dollar Lost values:")
    smallest_10 = df_finite.nsmallest(10, 'Dollar_Lost_processed')
    print(smallest_10[['Dollar_Lost_processed', 'tau_p_source']].to_string(index=False))

📊 CREATING VISUALIZATIONS FOR ZERO DOLLAR LOST COMBINATIONS
📈 Creating distribution plots for 7 variable parameters...



📋 Fixed Parameter Values (Required for Zero Dollar Lost):


## Summary

This notebook analyzes the parametric study results to identify parameter combinations that result in zero dollar lost.

### Key Findings:
- **Zero combinations found**: Check the output above
- **Fixed parameters**: Parameters that must have specific values for zero loss
- **Variable parameters**: Parameters that can vary while maintaining zero loss

### Next Steps:
1. If zero combinations are found, use the fixed parameter values as design constraints
2. If no exact zeros, analyze the smallest loss combinations
3. Use the parameter ranges to guide optimization efforts

### Usage:
Run all cells from top to bottom to perform the complete analysis.